# Pydantic Tutorial

This notebook teaches the basics of Pydantic with short explanations followed by runnable examples.

## Example 1: Creating a Simple Model

A Pydantic model is created by extending `BaseModel`. Pydantic automatically validates the data types and can even convert compatible values.

In [ ]:
from pydantic import BaseModel

class User(BaseModel):
    name: str
    age: int

user = User(name="John", age="25")

print(user)

## Example 2: Restricting Values with Literal

`Literal` allows a field to accept only specific values. This is useful when a field should be limited to a predefined list of options.

In [ ]:
from typing import Literal
from pydantic import BaseModel

class Ticket(BaseModel):
    priority: Literal["Low", "Medium", "High"]

ticket = Ticket(priority="High")

print(ticket)

In [ ]:
# This will not work
ticket = Ticket(priority="Higher")

## Example 3: Building a Real-World Model

This example creates a regulatory intake record. The model contains multiple fields, each with validation rules using `Literal`.

In [4]:
from typing import Literal                      # Used to restrict fields to a fixed set of allowed values
from pydantic import BaseModel, field_validator # BaseModel creates a data model, field_validator adds custom validation


# Define the schema for an intake record
class IntakeRecord(BaseModel):

    # Allowed query types
    query_type: Literal[
        "complaint", "submission", "variation", "safety_signal",
        "label_update", "inspection", "general_enquiry"
    ]

    # Allowed regulatory references
    regulation_ref: Literal[
        "FDA_21CFR", "EMA_CTR", "ICH_E2A", "ICH_Q10",
        "CDSCO_NDC", "GxP_GMP", "GxP_GCP", "other"
    ]

    # Allowed product or business areas
    product_area: Literal[
        "oncology", "cardiovascular", "infectious_disease",
        "cmc", "clinical", "labelling", "general"
    ]

    # Allowed urgency levels
    urgency: Literal["routine", "standard", "urgent", "critical"]

    # Team submitting the request
    submitting_team: str


    # Validate that the team name is not blank
    @field_validator("submitting_team")
    @classmethod
    def team_must_not_be_empty(cls, v):

        # Reject empty or whitespace-only values
        if not v.strip():
            raise ValueError("submitting_team cannot be empty")

        # Remove extra spaces and convert to title case
        return v.strip().title()


    # Validate that the value looks like a team name rather than a person's name
    @field_validator("submitting_team")
    @classmethod
    def team_not_a_person(cls, v):

        # Reject values like "JOHN" (single uppercase word)
        if len(v.split()) == 1 and v[0].isupper() and v.isalpha():
            raise ValueError(
                "submitting_team must be a team or function."
            )

        # Return the validated value
        return v

## Example 4: Creating a Valid Object

A dictionary containing valid data is passed to the model. Pydantic validates the input and creates a strongly typed object.

In [5]:
valid_data = {
    "query_type": "safety_signal",
    "regulation_ref": "ICH_E2A",
    "product_area": "clinical",
    "urgency": "critical",
    "submitting_team": "PV Team"
}

record = IntakeRecord(**valid_data)

print(record)
print(record.model_dump())

query_type='safety_signal' regulation_ref='ICH_E2A' product_area='clinical' urgency='critical' submitting_team='Pv Team'
{'query_type': 'safety_signal', 'regulation_ref': 'ICH_E2A', 'product_area': 'clinical', 'urgency': 'critical', 'submitting_team': 'Pv Team'}


In [6]:
invalid_data = {
    "query_type": "safety_signal",
    "regulation_ref": "ICH_E2A",
    "product_area": "clinical",
    "urgency": "critical",
    "submitting_team": "JOHN"
}

record = IntakeRecord(**invalid_data)

ValidationError: 1 validation error for IntakeRecord
submitting_team
  Value error, submitting_team must be a team or function. [type=value_error, input_value='JOHN', input_type=str]
    For further information visit https://errors.pydantic.dev/2.11/v/value_error

## Example 5: Handling Validation Errors

If invalid data is supplied, Pydantic raises a `ValidationError`. This example demonstrates how to capture and display detailed error messages.

In [7]:
from pydantic import ValidationError

try:
    bad_record = IntakeRecord(
        query_type="complaint",
        regulation_ref="WHO_GMP",
        product_area="clinical",
        urgency="critical",
        submitting_team="PV Team"
    )
except ValidationError as e:
    print("Validation failed:")

    for err in e.errors():
        print(
            f"Field: {err['loc'][0]} | Error: {err['msg']}"
        )

print("Moving on to next line of code....")        

Validation failed:
Field: regulation_ref | Error: Input should be 'FDA_21CFR', 'EMA_CTR', 'ICH_E2A', 'ICH_Q10', 'CDSCO_NDC', 'GxP_GMP', 'GxP_GCP' or 'other'
Moving on to next line of code....


## Summary

* `BaseModel` defines the structure of data.
* Type hints provide automatic validation.
* `Literal` restricts values.
* Validators implement custom business rules.
* `ValidationError` handles invalid data.
* `model_dump()` converts the model to a dictionary.